In [5]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix
)



# 1. DATA LOAD


data = pd.read_csv("hospital_readmission.csv")

print("Dataset Shape:", data.shape)

print("\nFirst 5 Rows:")
print(data.head())



# 2. PRE-PROCESSING / DATA CLEANING


# Separate features (X) and target (y)

X = data.drop("readmitted_30days", axis=1)

y = data["readmitted_30days"]


# Identify numerical and categorical features

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()


# Treat discharge_type as categorical
# because its numbers represent categories

if "discharge_type" in numerical_features:
    numerical_features.remove("discharge_type")
    categorical_features.append("discharge_type")


print("\nNumerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)



# Numerical preprocessing


numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])



# Categorical preprocessing

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])



# Combine both preprocessing pipelines


preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])



# 3. TRAIN-TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print("\nTraining Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)



# 4. ML MODEL



# MODEL 1: WITHOUT L2 REGULARIZATION


model_without_l2 = LogisticRegression(
    penalty=None,
    max_iter=1000
)


pipeline_without_l2 = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model_without_l2)
])



# MODEL 2: WITH L2 REGULARIZATION


model_with_l2 = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=1000
)


pipeline_with_l2 = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model_with_l2)
])



# 5. TRAINING


print("\nTraining model WITHOUT L2...")

pipeline_without_l2.fit(
    X_train,
    y_train
)


print("Training model WITH L2...")

pipeline_with_l2.fit(
    X_train,
    y_train
)



# 6. PREDICTION PROBABILITIES


# Probability of class 1

prob_without_l2 = (
    pipeline_without_l2.predict_proba(X_test)[:, 1]
)

prob_with_l2 = (
    pipeline_with_l2.predict_proba(X_test)[:, 1]
)



# 7. CONVERT PROBABILITIES INTO CLASSES


# Threshold = 0.5

prediction_without_l2 = (
    prob_without_l2 >= 0.5
).astype(int)


prediction_with_l2 = (
    prob_with_l2 >= 0.5
).astype(int)



# 8. ROC-AUC

auc_without_l2 = roc_auc_score(
    y_test,
    prob_without_l2
)


auc_with_l2 = roc_auc_score(
    y_test,
    prob_with_l2
)


print("\n======================================")
print("ROC-AUC RESULTS")
print("======================================")

print("ROC-AUC WITHOUT L2:", auc_without_l2)

print("ROC-AUC WITH L2:", auc_with_l2)



# 9. CONFUSION MATRIX

cm_without_l2 = confusion_matrix(
    y_test,
    prediction_without_l2
)


cm_with_l2 = confusion_matrix(
    y_test,
    prediction_with_l2
)


print("\n======================================")
print("CONFUSION MATRIX - WITHOUT L2")
print("======================================")

print(cm_without_l2)


print("\n======================================")
print("CONFUSION MATRIX - WITH L2")
print("======================================")

print(cm_with_l2)



# 10. FALSE NEGATIVE


tn1, fp1, fn1, tp1 = cm_without_l2.ravel()

tn2, fp2, fn2, tp2 = cm_with_l2.ravel()


print("\n======================================")
print("FALSE NEGATIVE RESULTS")
print("======================================")

print("False Negatives WITHOUT L2:", fn1)

print("False Negatives WITH L2:", fn2)


# ============================================================
# 11. CLASSIFICATION REPORT
# ============================================================

print("\n======================================")
print("CLASSIFICATION REPORT - WITHOUT L2")
print("======================================")

print(
    classification_report(
        y_test,
        prediction_without_l2
    )
)


print("\n======================================")
print("CLASSIFICATION REPORT - WITH L2")
print("======================================")

print(
    classification_report(
        y_test,
        prediction_with_l2
    )
)

Dataset Shape: (32300, 10)

First 5 Rows:
    age  length_of_stay_days  num_diagnoses  num_medications  prev_admissions  \
0  22.0                 28.0            2.0             13.0              4.0   
1  44.0                 12.0            7.0             11.0              2.0   
2  34.0                 15.0            9.0             16.0              6.0   
3  39.0                 15.0            5.0             16.0              6.0   
4  62.0                 24.0            1.0              6.0              6.0   

   glucose_level   bmi  has_diabetes  discharge_type  readmitted_30days  
0           89.1  48.4           1.0             2.0                  0  
1          254.8  23.9           1.0             0.0                  1  
2          270.2  16.7           0.0             1.0                  1  
3          208.6  18.2           0.0             1.0                  1  
4          233.1  44.0           0.0             0.0                  1  

Numerical Features:
['leng